In [1]:
import os
import seaborn as sns
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [301]:
## RUN THIS CELL ## 
from hbn.constants import Defaults
from hbn.data import make_dataset

# make_dataset.make_train_test_splits(out_dir=Defaults.MODEL_SPEC_DIR)

# get summary of clinical diagnosis + other demographics
dx = make_dataset.make_summary(save=False)
dx = make_dataset._add_race_ethnicity(dataframe=dx)

# filter for adhd
adhd_only = ['ADHD-Combined Type', 'ADHD-Hyperactive/Impulsive Type', 'ADHD-Inattentive Type', 'No Diagnosis Given']
dx = dx[dx['DX_01'].isin(adhd_only)]

### Basic Demographics

### What is the race breakdown of children with adhd? 

In [302]:
demographics = dx.groupby(['DX_01', 'PreInt_Demos_Fam,Child_Race_cat']
                          ).agg({'Identifiers': 'count',
                                }).reset_index()

fig = px.bar(demographics, x="DX_01", y="Identifiers", color="PreInt_Demos_Fam,Child_Race_cat")
fig.show()

### What is the sex breakdown of children with adhd? 
#### largest M/F ratios are combined and hyperactive, but not inattentive

In [303]:
demographics = dx.groupby(['DX_01', 'Sex']
                          ).agg({'Identifiers': 'count',
                                }).reset_index()

fig = px.bar(demographics, x="DX_01", y="Identifiers", color="Sex")
fig.show()

### What is the age breakdown of children with adhd? 

In [304]:
demographics = dx.groupby(['DX_01', 'Age_bracket']
                          ).agg({'Identifiers': 'count',
                                }).reset_index()

fig = px.bar(demographics, x="DX_01", y="Identifiers", color="Age_bracket")
fig.show()

### How many comorbidities do children with adhd have?
#### Girls have more combordities on average than boys (except for impulsive type)

In [305]:
demographics = dx.groupby(['DX_01', 'Sex']
                          ).agg({'comorbidities': 'mean',
                                }).reset_index()

fig = px.bar(demographics, x="DX_01", y="comorbidities", color='Sex')
fig.show()

### How many comorbidities do children with adhd have?
#### Childrenn over10 have more combordities on average than children under10 

In [306]:
demographics = dx.groupby(['DX_01', 'Age_bracket']
                          ).agg({'comorbidities': 'mean',
                                }).reset_index()

fig = px.bar(demographics, x="DX_01", y="comorbidities", color='Age_bracket')
fig.show()

In [210]:
# interactive data widget
from hbn.data import data_widgets

phenotype_widget = data_widgets.get_phenotype_inputs()
display(phenotype_widget)

participant_widget = data_widgets.get_participant_inputs()
display(participant_widget)

### What % of children with adhd have parents with adhd?

In [307]:
from hbn.data import make_dataset

### INTAKE INTERVIEW ###

# get features and target
features, target = data_widgets.get_data(phenotype_widget, participant_widget, preprocess=False)

# create dataframe (merging features + target)
df_intake = pd.concat([pd.DataFrame(target), features], axis=1)
df_intake = df_intake.merge(dx[['Sex', 'Age_bracket', 'PreInt_Demos_Fam,Child_Race_cat','Identifiers']], on='Identifiers')

tmp = df_intake.groupby('DX_01').agg({'PreInt_FamHx,m_adhd': 'sum', 
                               'PreInt_FamHx,f_adhd': 'sum', 
                               'PreInt_FamHx,s_adhd': 'sum',
                               'PreInt_FamHx,f_autism':'sum',
                               'PreInt_FamHx,m_autism':'sum',
                               'PreInt_FamHx,s_autism':'sum',
                               'Identifiers': 'count'})
tmp

reading /Users/maedbhking/Documents/healthy_brain_network/data/raw/phenotype/Parent_Measures/Interview_of_Emotional_and_Psychological_Function/Intake_Interview.csv into dataframe
reading /Users/maedbhking/Documents/healthy_brain_network/data/raw/phenotype/Clinical_Measures/Clinical_Diagnosis_Demographics.csv into dataframe


,"PreInt_FamHx,m_adhd","PreInt_FamHx,f_adhd","PreInt_FamHx,s_adhd","PreInt_FamHx,f_autism","PreInt_FamHx,m_autism","PreInt_FamHx,s_autism",Identifiers
DX_01,,,,,,,
ADHD-Combined Type,3.0,6.0,4.0,0.0,0.0,3.0,753
ADHD-Hyperactive/Impulsive Type,0.0,0.0,0.0,0.0,0.0,1.0,106
ADHD-Inattentive Type,0.0,1.0,2.0,0.0,0.0,2.0,680
No Diagnosis Given,1.0,4.0,4.0,0.0,0.0,1.0,332


In [219]:
df_intake.columns.str.split(',').str.get(0).unique()

Index(['DX_01', 'Identifiers', 'PreInt_Demos_Fam', 'PreInt_Demos_Home',
       'PreInt_DevHx', 'PreInt_EduHx', 'PreInt_FamHx', 'PreInt_FamHx_RDC',
       'PreInt_Lang', 'PreInt_TxHx'],
      dtype='object')

### Previous diagnoses
#### Many children with adhd have been previously diagnosed with a psych/learning disorder

In [308]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat']

for color in colors:
    tmp = df_intake.groupby(['DX_01', color]).agg({'PreInt_TxHx,Past_DX': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp['PreInt_TxHx,Past_DX'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent', color=color,orientation='v', barmode="group")
    fig.show()

### about 25% of children with adhd are currently taking psych medication

In [309]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat']

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({'PreInt_TxHx,psych_meds_cur': 'sum',
                                            'PreInt_TxHx,psych_meds_past': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent_curr'] = tmp['PreInt_TxHx,psych_meds_cur'] / tmp['Identifiers']
    tmp['percent_past'] = tmp['PreInt_TxHx,psych_meds_past'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent_curr', color=color,orientation='v', barmode="group")
    fig.show()

### few children had immunication reactions

In [310]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat']
var = 'immunization_reaction'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_TxHx,{var}': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_TxHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent', color=color,orientation='v', barmode="group")
    fig.show()

### 10-20% of children have had food allergies

In [311]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat']
var = 'food_allergy'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_TxHx,{var}': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_TxHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent', color=color,orientation='v', barmode="group")
    fig.show()

### Most children have attended an average of 2 schools

In [364]:
colors = ['Sex', 'Age_bracket']
var = 'number_schools'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_EduHx,{var}': 'mean',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_EduHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y=f'PreInt_EduHx,{var}', color=color,orientation='v', barmode="group")
    fig.update_yaxes(range=[1,4])
    fig.show()

### 50-60% of children have an individualized education plan
#### more children over10 with hyperactive/impulsive have an IEP but more children under10 with inattentive have an IEP

In [358]:
colors = ['Sex', 'Age_bracket']
var = 'IEP'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_EduHx,{var}': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_EduHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent', color=color,orientation='v', barmode="group")
    fig.update_yaxes(range=[.1,.7])
    fig.show()

### learning disability?
#### few children with adhd diagnosed with a learning disability

In [318]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat', 'Age_bracket']
var = 'learning_disability'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_EduHx,{var}': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_EduHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent', color=color,orientation='v', barmode="group")
    fig.show()

### neuropsych testing? pretty low numbers ...

In [314]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat', 'Age_bracket']
var = 'NeuroPsych'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_EduHx,{var}': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_EduHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent', color=color,orientation='v', barmode="group")
    fig.show()

### Recent grades (1-excellent, 5-failing)

In [355]:
colors = ['Sex','Age_bracket']
var = 'recent_grades'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_EduHx,{var}': 'mean',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_EduHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y=f'PreInt_EduHx,{var}', color=color,orientation='v', barmode="group")
    fig.update_yaxes(range=[1,3])
    fig.show()

### number of friends

In [352]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat', 'Age_bracket']
var = 'number_friends'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_EduHx,{var}': 'mean',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_EduHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y=f'PreInt_EduHx,{var}', color=color,orientation='v', barmode="group")
    fig.update_yaxes(range=[1,4])
    fig.show()

### outside school tutoring
#### 40% of children with inattentive type adhd have outside tutoring

In [319]:
colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat', 'Age_bracket']
var = 'tutor'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_EduHx,{var}': 'sum',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_EduHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y='percent', color=color,orientation='v', barmode="group")
    fig.show()

### start of puberty
#### girls with adhd are starting puberty a lot earlier than boys - this tracks with children without a diagnosis. exception is boys with hyperactive adhd

In [350]:

colors = ['Sex', 'PreInt_Demos_Fam,Child_Race_cat']
var = 'puberty_age'

for color in colors:
    tmp = df_intake.groupby(['DX_01',color]).agg({f'PreInt_DevHx,{var}': 'mean',
                                   'Identifiers': 'count'}
                                        ).reset_index()
    tmp['percent'] = tmp[f'PreInt_DevHx,{var}'] / tmp['Identifiers']

    fig = px.bar(tmp, x="DX_01", y=f'PreInt_DevHx,{var}', color=color,orientation='v', barmode="group")
    fig.update_yaxes(range=[8,12])
    fig.show()

### girls with hyperactive/impulsive adhd are starting menstruation earlier than other subtypes

In [348]:
var = 'menstruation_age'

tmp = df_intake.groupby(['DX_01']).agg({f'PreInt_DevHx,{var}': 'mean',
                               'Identifiers': 'count'}
                                    ).reset_index()

fig = px.bar(tmp, x="DX_01", y=f'PreInt_DevHx,{var}',orientation='v', barmode="group")
fig.update_yaxes(range=[10,12])
fig.show()